# PySpark 5: writing, pipelines and quality

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-05-writing-pipelines-and-quality.ipynb)

A nightly job publishes order tables that finance trusts. It must replace only the days it rebuilt, apply
late changes exactly once, keep history for changing customer attributes, and give the same result when the
scheduler retries it.

This module writes tables with an explicit mode and scope, loads changes incrementally, builds a slowly
changing dimension, merges idempotently into a Delta table, tests transformations, quarantines bad records,
and chooses built-in functions over Python UDFs.

Each section explains how the feature works, where it matters and what to watch out for, then ends with a runnable example. Run it, read the output, and compare it with the notes on what to notice. Run the cells in order.

**Prerequisites:** [PySpark 4: windows and nested data](pyspark-04-windows-and-nested-data.ipynb).

*Last reviewed: November 2026, against Apache Spark 4.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0 pandas pyarrow deltalake

## Setup and sample data

The first cell starts a local Spark session. The second creates `orders` (one row per order; order 12 lands just after midnight UTC) and `order_items` (one row per order line; each order's lines add up to its amount): a slice of the orders platform that every
module shares, introduced in [PySpark 1](pyspark-01-foundations.ipynb#sample-data-used-in-every-module).

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import functions as F

orders = spark.createDataFrame([
    (1, 1, "paid", datetime(2026, 1, 5, 9, 15), D("120.00")),
    (2, 1, "shipped", datetime(2026, 1, 5, 10, 5), D("84.50")),
    (3, 1, "returned", datetime(2026, 1, 5, 11, 40), D("30.00")),
    (4, 2, "paid", datetime(2026, 1, 6, 15, 20), D("260.00")),
    (5, 2, "placed", datetime(2026, 1, 7, 8, 5), D("55.25")),
    (6, 3, "shipped", datetime(2026, 1, 7, 17, 45), D("149.00")),
    (7, 3, "paid", datetime(2026, 1, 9, 12, 10), D("310.00")),
    (8, 4, "returned", datetime(2026, 1, 10, 19, 30), D("89.90")),
    (9, 4, "shipped", datetime(2026, 1, 12, 8, 55), D("1180.00")),
    (10, 5, "paid", datetime(2026, 1, 12, 13, 0), D("74.00")),
    (11, 6, "placed", datetime(2026, 1, 13, 10, 25), D("210.00")),
    (12, 6, "shipped", datetime(2026, 1, 14, 0, 30), D("96.40")),  # just after midnight UTC
], "order_id int, customer_id int, status string, order_ts timestamp, amount decimal(12,2)")

order_items = spark.createDataFrame([
    (1, "SKU-4001", 2, D("12.50")), (1, "SKU-4007", 1, D("95.00")),
    (2, "SKU-4002", 1, D("34.50")), (2, "SKU-4009", 2, D("25.00")),
    (3, "SKU-4002", 1, D("30.00")),
    (4, "SKU-4003", 1, D("149.00")), (4, "SKU-4004", 2, D("55.50")),
    (5, "SKU-4005", 1, D("55.25")),
    (6, "SKU-4003", 1, D("149.00")),
    (7, "SKU-4006", 1, D("189.00")), (7, "SKU-4008", 1, D("121.00")),
    (8, "SKU-4010", 1, D("89.90")),
    (9, "SKU-4011", 20, D("59.00")),  # a bulk reorder
    (10, "SKU-4012", 2, D("37.00")),
    (11, "SKU-4013", 1, D("210.00")),
    (12, "SKU-4014", 1, D("41.40")), (12, "SKU-4005", 1, D("55.00")),
], "order_id int, sku string, qty int, unit_price decimal(8,2)")

In [ ]:
import contextlib
import io


def plan_of(df, mode="formatted"):
    """Return the text `df.explain(mode)` prints, so a cell can pick out the lines that matter."""
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        df.explain(mode)
    return buffer.getvalue()

## Write tables with an explicit mode

A write needs a destination and a **save mode**:

| mode | when the table exists |
| --- | --- |
| `errorifexists` (default) | the write fails |
| `append` | rows are added |
| `overwrite` | the table (or, with dynamic overwrite, the touched partitions) is replaced |
| `ignore` | nothing happens, silently |

Prefer catalog table names (`catalog.schema.table` on Unity Catalog) over raw paths, so permissions,
lineage and the schema live in the catalog. `saveAsTable` is the v1 writer; `df.writeTo("t").append()` and
`.overwritePartitions()` are the v2 API for Delta and Iceberg catalogs. `partitionBy("order_date")` writes one
directory per value.

**Where it matters**
- **Gold tables** rebuilt from silver on every run: `overwrite`.
- **Append-only logs** such as pipeline metrics or quarantine records: `append`.
- **First-time setup jobs** that must never clobber data: `errorifexists`.

**Watch out for**
- `append` on a retried job duplicates data unless the batch is deduplicated against the target.
- `ignore` hides the fact that nothing was written. Use it only when skipping really is the intent.
- Partitioning by a high-cardinality column such as `customer_id` creates millions of tiny directories. Partition by a date most queries filter on.
- `overwriteSchema` accepts upstream type drift and changes the table contract. Own the schema in the pipeline and fail fast instead.

### Example

Build daily product sales, write them with each mode, and check what each did. Then write a copy
partitioned by `order_date`.

In [ ]:
from pyspark.errors import AnalysisException

daily_product_sales = (
    order_items
    .join(orders.select("order_id", "order_ts"), "order_id")
    .withColumn("order_date", F.to_date("order_ts"))
    .groupBy("order_date", "sku")
    .agg(F.sum("qty").alias("units"), F.sum(F.col("qty") * F.col("unit_price")).alias("revenue"))
)
rows = daily_product_sales.count()

spark.sql("create database if not exists gold")
daily_product_sales.write.mode("overwrite").saveAsTable("gold.daily_product_sales")
daily_product_sales.write.mode("ignore").saveAsTable("gold.daily_product_sales")  # exists: skipped
print("after overwrite + ignore:", spark.table("gold.daily_product_sales").count(), "rows")

try:
    daily_product_sales.write.mode("errorifexists").saveAsTable("gold.daily_product_sales")
except AnalysisException as exc:
    print("errorifexists refused:", exc.getCondition())
else:
    raise AssertionError("errorifexists should refuse an existing table")

daily_product_sales.write.mode("append").saveAsTable("gold.daily_product_sales")
print("after append          :", spark.table("gold.daily_product_sales").count(), "rows")

daily_product_sales.write.mode("overwrite").partitionBy("order_date").saveAsTable("gold.daily_product_sales_by_day")
spark.sql("show partitions gold.daily_product_sales_by_day").show()

**What to notice:** `ignore` changed nothing and said nothing. `errorifexists` refused with `TABLE_OR_VIEW_ALREADY_EXISTS`.
`append` doubled the row count: exactly what a retried append job does. The partitioned copy has one
`order_date=...` directory per day.

## Overwrite only the partitions you rebuild

On a partitioned table, `mode("overwrite")` replaces the **whole** table unless
`spark.sql.sources.partitionOverwriteMode` is `dynamic`. In dynamic mode only the partitions present in the
DataFrame are replaced, and every other partition keeps its files. Delta and Iceberg give the same scope with
`replaceWhere` or `writeTo(...).overwritePartitions()`.

**Where it matters**
- **Reprocessing one day** after a bug fix without touching the other 364.
- **Backfills** that rebuild a month: one write replaces exactly those dates.

**Watch out for**
- A date whose correct result is now empty has no rows, so dynamic overwrite never replaces it. Delete it explicitly.
- Static overwrite with a filtered DataFrame deletes every partition you did not include. Set the mode deliberately in every job.
- A rewritten partition must keep the table's column types. Plain Parquet does not check, so one partition written as `double` breaks every later read of the table.

### Example

A pricing bug on 12 January is fixed. Rewrite only that day and check that the other days' revenue and
partitions are unchanged.

In [ ]:
sales_path = os.path.join(WORK_DIR, "daily_product_sales")
daily_product_sales.write.mode("overwrite").partitionBy("order_date").parquet(sales_path)
other_days = "order_date != '2026-01-12'"
before = spark.read.parquet(sales_path).where(other_days).agg(F.sum("revenue")).first()[0]

revenue_type = daily_product_sales.schema["revenue"].dataType  # keep the partition's type identical
corrected = (
    daily_product_sales.where("order_date = '2026-01-12'")
    .withColumn("revenue", (F.col("revenue") * F.lit(D("0.9"))).cast(revenue_type))
)
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")
try:
    corrected.write.mode("overwrite").partitionBy("order_date").parquet(sales_path)
finally:
    spark.conf.set("spark.sql.sources.partitionOverwriteMode", "static")

after = spark.read.parquet(sales_path)
print("other days' revenue before:", before, "| after:", after.where(other_days).agg(F.sum("revenue")).first()[0])
after.groupBy("order_date").agg(F.sum("revenue").alias("revenue")).orderBy("order_date").show()

**What to notice:** Every date is still present, and the other days' revenue is unchanged: only the `2026-01-12` directory was
replaced, now with revenue 10 percent lower. With the default static mode, the same write would have
deleted every other day. The correction is cast back to the column's original decimal type: multiplying
by the float `0.9` would write that one partition as `double`, and reading the whole table would then
fail on the mismatched files.

## Load only new changes

An incremental job processes only what arrived since the last successful run. Keep the high-water mark (a
timestamp or source sequence) in a small operations table, read only newer source rows, keep the latest
change per key, and upsert them into the target. Delta and Iceberg do the upsert with `MERGE`:

```sql
merge into main.silver.orders as target
using staged_orders as source
on target.order_id = source.order_id
when matched then update set *
when not matched then insert *
```

Plain Parquet has no `MERGE`, so the example expresses the same upsert as "target rows whose key did not
change, plus the staged rows", and runs it twice to prove it is idempotent.

**Where it matters**
- **Hourly loads** from a landing zone or a CDC feed into a silver table.
- **Large targets:** merging 5 million changes into 2 billion rows rewrites only the files that hold matched keys.

**Watch out for**
- Deriving the watermark from `max(ingested_at)` of the target. Late rows that arrive after that value are skipped forever; keep the watermark in an operations table with a deliberate lookback.
- A source batch with two rows for one key makes `MERGE` fail or pick arbitrarily. Deduplicate the batch first.
- Advance the watermark only after the target write commits.

### Example

The last run loaded orders up to 10 January. Take newer orders plus a refund for order 8 that arrived twice,
keep one change per key, upsert, and run the upsert again.

In [ ]:
last_loaded_ts = datetime(2026, 1, 10, 0, 0)  # read from an operations table in a real job

silver = orders.where(F.col("order_ts") <= F.lit(last_loaded_ts))  # what the last run loaded
change_schema = "order_id int, customer_id int, status string, order_ts timestamp, amount decimal(12,2), source_lsn int, ingested_at timestamp"
bronze_changes = (
    orders.where(F.col("order_ts") > F.lit(last_loaded_ts))
    .withColumn("source_lsn", F.lit(1))
    .withColumn("ingested_at", F.col("order_ts"))
    .unionByName(spark.createDataFrame(  # order 8 was refunded, and the change arrived twice
        [(8, 4, "refunded", datetime(2026, 1, 10, 19, 30), D("89.90"), 2, datetime(2026, 1, 15, 8, 0)),
         (8, 4, "refunded", datetime(2026, 1, 10, 19, 30), D("89.90"), 2, datetime(2026, 1, 15, 8, 5))],
        change_schema,
    ))
)

from pyspark.sql import Window

latest_change = Window.partitionBy("order_id").orderBy(F.desc("source_lsn"), F.desc("ingested_at"))
staged = (
    bronze_changes.withColumn("rn", F.row_number().over(latest_change))
    .where("rn = 1")
    .drop("rn", "source_lsn", "ingested_at")
)


def upsert(target, changes, key="order_id"):
    untouched = target.join(changes.select(key), key, "left_anti")
    return untouched.unionByName(changes)

once = upsert(silver, staged)
twice = upsert(once, staged)
once.orderBy("order_id").show()
print("rows that differ between one and two runs:", once.exceptAll(twice).count() + twice.exceptAll(once).count())

**What to notice:** Silver has 12 orders, one per key: order 8's refund arrived twice but was staged once, and the newer
orders were added. Applying the same staged batch a second time changes nothing, so a retry is harmless.

## Keep history with a slowly changing dimension

A **slowly changing dimension type 2** (SCD2) keeps every version of an attribute. Each row has `valid_from`,
`valid_to` (null while current) and `is_current`. When a change arrives, the current row is closed at the
change date and a new current row is opened; unchanged keys are left alone, and new keys are inserted.
Facts then join to the version that was valid at the event time.

On Delta or Iceberg this is one `MERGE` with a staged union of "rows to close" and "rows to insert". The
example builds it with DataFrames so every step is visible.

**Where it matters**
- **Customer region or segment history,** so last year's revenue stays attributed to last year's region.
- **Audits:** answering "what did we believe on 1 March?"

**Watch out for**
- Compare attributes with null-safe equality; a change from null to a value is a change.
- Exactly one current row per key is the invariant. Assert it after every load.
- Rerunning the same change batch must not open a second version. Compare against the current value, not just the key.

### Example

Customer 1 moves from North to West, customer 2 sends an unchanged record, and customer 3 is new. Apply the
changes, check the invariants, then apply them again and check nothing changed.

In [ ]:
dim_schema = "customer_id int, region string, valid_from date, valid_to date, is_current boolean"
dim = spark.createDataFrame(
    [(1, "North", date(2024, 1, 3), None, True), (2, "North", date(2024, 2, 11), None, True)], dim_schema
)
changes = spark.createDataFrame(
    [(1, "West", date(2026, 1, 15)), (2, "North", date(2026, 1, 15)), (3, "South", date(2026, 1, 15))],
    "customer_id int, region string, changed_on date",
)


def apply_scd2(dim, changes):
    current = dim.where("is_current").select("customer_id", F.col("region").alias("current_region"))
    changed = (
        changes.join(current, "customer_id", "left")
        .where(~F.col("region").eqNullSafe(F.col("current_region")))  # new keys and new values
    )
    closing = changed.select("customer_id", "changed_on")
    closed = (
        dim.join(closing, "customer_id", "left")
        .withColumn("closes", F.col("is_current") & F.col("changed_on").isNotNull())
        .withColumn("valid_to", F.when(F.col("closes"), F.col("changed_on")).otherwise(F.col("valid_to")))
        .withColumn("is_current", F.col("is_current") & ~F.col("closes"))
        .drop("changed_on", "closes")
    )
    opened = changed.select(
        "customer_id", "region", F.col("changed_on").alias("valid_from"),
        F.lit(None).cast("date").alias("valid_to"), F.lit(True).alias("is_current"),
    )
    return closed.unionByName(opened)

dim_v2 = apply_scd2(dim, changes)
dim_v2.orderBy("customer_id", "valid_from").show()

current_rows = dim_v2.groupBy("customer_id").agg(F.sum(F.col("is_current").cast("int")).alias("current_rows"))
current_rows.orderBy("customer_id").show()  # the invariant: exactly one current row per key

dim_v3 = apply_scd2(dim_v2, changes)  # the same batch again
print("rows changed by a rerun:", dim_v3.exceptAll(dim_v2).count() + dim_v2.exceptAll(dim_v3).count())

**What to notice:** Customer 1 now has two rows: North, closed on 2026-01-15, and West, current from that date. Customer 2's
unchanged record opened nothing, and customer 3 is new. Every customer has exactly one current row, and
rerunning the batch changes nothing because each change is compared with the current value.

## Make merges idempotent and version-aware

An **idempotent** write leaves the same result when the same batch is applied twice. A job can commit its
output and then fail before the scheduler records success; the retry must not duplicate or regress data.
With a transactional table this means: a stable business key, at most one source row per key per batch,
and an update condition that only lets a **newer** source version win, so an old delivery replayed late
cannot overwrite a newer row.

In PySpark with the `delta-spark` package the merge reads:

```python
from delta.tables import DeltaTable

(
    DeltaTable.forName(spark, "main.silver.orders").alias("t")
    .merge(staged.alias("s"), "t.order_id = s.order_id")
    .whenMatchedUpdateAll(condition="s.source_version > t.source_version")
    .whenNotMatchedInsertAll()
    .execute()
)
```

`delta-spark` needs Delta's JARs on the Spark classpath. To run without downloading them, the example uses
the [`deltalake`](https://delta-io.github.io/delta-rs/api/delta_table/) Python package, which writes the same
Delta format and has the same merge semantics.

**Where it matters**
- **Retried batches** after a failure between commit and success reporting.
- **Out-of-order deliveries:** a vendor resends Monday's file on Wednesday, after Tuesday's corrections.

**Watch out for**
- Deduplicate each source batch with an ordering that survives replay before merging.
- Without a version condition, a replayed old batch silently overwrites newer values.
- Keep the original input so you can reconcile and replay.

### Example

Order 1 starts at source version 1. A correction at version 2 raises its amount from 10 to 15. Apply the
correction twice, then replay the original delivery: the newer version must survive, and Delta's version 0
must still be readable through time travel.

In [ ]:
import pyarrow as pa
from deltalake import DeltaTable, write_deltalake


def merge_orders(table, changes):
    (
        table.merge(changes, predicate="t.id = s.id", source_alias="s", target_alias="t")
        .when_matched_update_all(predicate="s.source_version > t.source_version")
        .when_not_matched_insert_all()
        .execute()
    )

with tempfile.TemporaryDirectory(prefix="delta-merge-") as directory:
    initial = pa.table({"id": [1, 2], "amount": [10, 20], "source_version": [1, 1]})
    write_deltalake(directory, initial)
    table = DeltaTable(directory)
    table.delete(predicate="id = 2")
    correction = pa.table({"id": [1, 3], "amount": [15, 30], "source_version": [2, 1]})

    merge_orders(table, correction)
    merge_orders(table, correction)  # a retry
    merge_orders(table, initial.slice(0, 1))  # the stale version-1 delivery, replayed late

    print("current:", sorted(table.to_pyarrow_table().to_pylist(), key=lambda row: row["id"]))
    print("version 0:", DeltaTable(directory, version=0).to_pyarrow_table().to_pylist())  # time travel
    print("table versions:", [entry["operation"] for entry in reversed(table.history())])

**What to notice:** Order 1 ends at amount 15, source version 2. The retry matched but changed nothing, and the replayed
version-1 delivery matched but failed the `s.source_version > t.source_version` condition, so it could not
overwrite the newer row. The history shows a single `MERGE`: the two no-op merges committed no new table
version. Version 0 is still readable through time travel.

## Test transformations with small DataFrames

Keep business rules in functions that take and return DataFrames, then test them with a few hand-written
rows. Compare **rows** without depending on order, and compare the **schema** too: a rule that changes a
type is as broken as one that changes a value. In a project, run the tests with `pytest` and one shared
local `SparkSession` fixture.

PySpark ships `pyspark.testing.assertDataFrameEqual`, which does both comparisons and prints a readable diff:

```python
from pyspark.testing import assertDataFrameEqual

assertDataFrameEqual(add_order_date(input_df), expected)  # ignores row order by default
```

On Spark 4.0 it needs pandas older than 3.0, because it imports pandas-on-Spark internals that pandas 3
removed. The example uses a three-line helper so it runs with any pandas version.

**Where it matters**
- **Pull requests:** every rule change ships with a test that covers nulls, duplicate keys, late data and boundary timestamps.
- **Refactors:** a test pins the output while you rewrite the implementation.

**Watch out for**
- Comparing `collect()` output without sorting. Spark does not guarantee row order, so the test passes locally and fails in CI.
- Testing only the happy path. Most production bugs are nulls, duplicates and time zones.

### Example

Test that `add_order_date` derives the date, keeps nulls and returns a `date` column; then check that a wrong expectation fails.

In [ ]:
def assert_frames_equal(actual, expected):
    """Same schema and the same rows, in any order."""
    assert actual.schema.simpleString() == expected.schema.simpleString(), (actual.schema, expected.schema)
    assert sorted(actual.collect(), key=str) == sorted(expected.collect(), key=str)


def add_order_date(df):
    return df.withColumn("order_date", F.to_date("order_ts"))

input_df = spark.createDataFrame(
    [("o1", "2026-10-04 23:30:00"), ("o2", None)], "order_id string, order_ts string"
)
expected = spark.createDataFrame(
    [("o2", None), ("o1", date(2026, 10, 4))], "order_id string, order_date date"  # order does not matter
)
assert_frames_equal(add_order_date(input_df).select("order_id", "order_date"), expected)

wrong = spark.createDataFrame([("o1", date(2026, 10, 5)), ("o2", None)], "order_id string, order_date date")
try:
    assert_frames_equal(add_order_date(input_df).select("order_id", "order_date"), wrong)
except AssertionError:
    print("the wrong expectation fails, as a test should")
else:
    raise AssertionError("the wrong expectation should fail")

## Quarantine bad records and account for them

Bad records should be isolated, counted and visible. Evaluate every rule once into a reason column, then
split the frame into valid rows and a quarantine table that keeps the reason and a timestamp. Accepted plus
rejected rows must equal input rows; log those counts for every run.

**Where it matters**
- **Before publishing gold:** a dashboard should never discover broken data before the pipeline does.
- **Vendor feedback:** the quarantine table, grouped by reason, is the report you send back to the producer.

**Watch out for**
- Dropping bad rows silently. Do it only when the data owner agrees that loss is acceptable.
- Splitting with `subtract` after adding a `rejected_at` column. The columns no longer match, so nothing cancels out. Filter on a reason column instead.
- Both writes read the checked frame; cache it, or the source is read and checked twice.

### Example

Check five orders against three rules, write valid rows and the quarantine, and reconcile the counts.

In [ ]:
bronze_orders_raw = spark.createDataFrame(
    [(1, 10, D("50.00")), (2, None, D("20.00")), (None, 11, D("5.00")), (4, 12, D("-3.00")), (5, 13, D("8.00"))],
    "order_id int, customer_id int, amount decimal(12,2)",
)

checked = bronze_orders_raw.withColumn(
    "quality_error",
    F.when(F.col("order_id").isNull(), "missing_order_id")
    .when(F.col("customer_id").isNull(), "missing_customer_id")
    .when(F.col("amount") < 0, "negative_amount"),
).cache()

valid_orders = checked.where("quality_error is null").drop("quality_error")
quarantine = checked.where("quality_error is not null").withColumn("rejected_at", F.current_timestamp())
valid_orders.write.mode("overwrite").saveAsTable("silver_orders_checked")
quarantine.write.mode("overwrite").saveAsTable("ops_orders_quarantine")

quarantine.groupBy("quality_error").count().orderBy("quality_error").show()
print("input:", bronze_orders_raw.count(), "| valid:", valid_orders.count(), "| quarantined:", quarantine.count())
checked.unpersist()

**What to notice:** Each rejected row carries one reason, the first rule it broke. Input rows equal valid plus quarantined rows,
so nothing disappeared; that line is what the pipeline should log on every run.

## Choose built-in functions before a Python UDF

A row-at-a-time Python UDF plans as a `BatchEvalPython` node: each task starts a Python worker, ships rows to
it and reads results back. Catalyst cannot see inside it, so filters do not push through it and code
generation stops there. Prefer, in order: a built-in function, a SQL expression or higher-order function, a
vectorized **pandas UDF** (Arrow batches as `pandas.Series`), and only then a row UDF.

`mapInPandas` and `groupBy(...).applyInPandas(...)` run a pandas function over whole partitions or groups when
the logic needs a DataFrame, not a column.

**Where it matters**
- **Email, phone and date parsing:** almost always expressible with built-ins.
- **Numerical or ML scoring** that needs NumPy or a model: a pandas UDF.

**Watch out for**
- A UDF closure that captures a large lookup dict is pickled into every task. Join to a broadcast table instead.
- A Series-to-Series pandas UDF is called once per Arrow batch, so `s.mean()` is the batch mean, not the column mean.
- Python UDFs need the same Python packages on every worker.

### Example

Extract an email domain with a built-in expression and with a Python UDF: same answer, but only the UDF plan
contains `BatchEvalPython`. Then cap a score with a vectorized pandas UDF.

In [ ]:
import pandas as pd
from pyspark.sql import types as T

raw_emails = spark.createDataFrame([("  Ada@Example.COM ",), ("not-an-email",), (None,)], "email string")


@F.udf(T.StringType())
def email_domain_udf(email):
    if email is None or "@" not in email:
        return None
    return email.strip().lower().split("@")[-1]

builtin = raw_emails.withColumn(
    "domain", F.when(F.col("email").contains("@"), F.element_at(F.split(F.lower(F.trim("email")), "@"), -1))
)
with_udf = raw_emails.withColumn("domain", email_domain_udf("email"))
builtin.show()
with_udf.show()
print("built-in plan crosses into Python:", "BatchEvalPython" in plan_of(builtin))
print("UDF plan crosses into Python     :", "BatchEvalPython" in plan_of(with_udf))


@F.pandas_udf("double")
def cap_score(score: pd.Series) -> pd.Series:
    return score.clip(lower=0.0, upper=1.0)

scores = spark.createDataFrame([(1, -0.2), (2, 0.4), (3, 1.7)], "user_id int, score double")
scores.withColumn("capped", cap_score("score")).show()

**What to notice:** Both versions give the same domains, but only the UDF's plan contains `BatchEvalPython`, the step that ships
rows to a Python worker and back. The pandas UDF receives a whole Arrow batch as a `pandas.Series` and clips
it in one vectorized call.

## Check yourself

Answer each question in your own words before reading the answer.

**Q1.** A retried job used `mode("append")`. What happened, and what should it use?

**A.** The batch was written twice. Use a write scoped to what the batch owns: dynamic partition overwrite, or a `MERGE` on the business key.

**Q2.** Dynamic partition overwrite is on. A corrected day now has no rows. What happens to its partition?

**A.** Nothing: no rows means no partition in the output, so the old data stays. Delete it explicitly.

**Q3.** Why keep the incremental watermark outside the target table?

**A.** A watermark derived from `max(ingested_at)` of the target skips late rows forever. An operations table with a lookback is advanced only after the write commits.

**Q4.** What invariant must an SCD2 dimension satisfy after every load?

**A.** Exactly one current row per key, with non-overlapping validity intervals. Rerunning the same batch must not open new versions.

**Q5.** How does a merge stay correct when an old delivery is replayed after a newer one?

**A.** The update condition only lets a newer source version win (`s.source_version > t.source_version`), so the stale rows match but do not update.

**Q6.** Why must a DataFrame test sort rows and compare the schema?

**A.** Spark does not guarantee row order, so an unsorted comparison fails intermittently. And a rule that changes a column's type is a bug even when the values look right.

**Q7.** When is a pandas UDF the right choice?

**A.** When the logic needs Python libraries such as NumPy or a model and no built-in function exists. It moves Arrow batches instead of single rows, but still crosses the JVM boundary.

## Key takeaways

- Choose the save mode and scope deliberately; rerunning a day must give the same table.
- Keep the watermark in an operations table, deduplicate each batch, and merge on the business key.
- SCD2: close the current row, open a new one, and assert one current row per key.
- Guard merges with a source-version condition so late replays cannot regress data.
- Test rules with small DataFrames, comparing rows in any order and the schema; quarantine bad rows with a reason.
- Built-in functions first, pandas UDFs second, row UDFs last.

**Next:** [PySpark 6: structured streaming](pyspark-06-structured-streaming.ipynb)